# Tornado Experiment — Master Runner

Executes the full analysis pipeline without visualizations:

1. **Setup** — model attributes, paths
2. **Data loading** — attribute tables + wide-format simulation output
3. **Post-processing** — intertemporal decomposition / rescaling
4. **Cost-benefits** — system + technical costs → `cost_benefits_data_tornado.csv`
5. **MAC analysis** — marginal abatement costs → `marginal_abatement_costs_tornado.csv`

All configuration lives in `scripts/config.py`.  
Shared pipeline modules (`model_setup`, `data_loading`, `postprocessing`,
`cost_benefits_pipeline`) are imported from `../shared_scripts/`.

## 0 · Environment setup

In [ ]:
import os
import sys
import pathlib
import logging
import warnings

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
RUNNER_DIR  = pathlib.Path(os.getcwd()).resolve()
SCRIPTS_DIR = RUNNER_DIR / "scripts"
SHARED_DIR  = RUNNER_DIR.parent / "shared_scripts"

assert SCRIPTS_DIR.exists(), f"scripts/ not found at {SCRIPTS_DIR}"
assert SHARED_DIR.exists(),  f"shared_scripts/ not found at {SHARED_DIR}"

for p in (SCRIPTS_DIR, SHARED_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# ── Config ────────────────────────────────────────────────────────────────────
import config as cfg

# Project root (needed for ssp_modeling.* imports inside pipeline scripts)
if str(cfg.PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(cfg.PROJECT_DIR))

# notebooks/ dir (needed for utils.logger_utils)
if str(cfg.NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(cfg.NOTEBOOKS_DIR))

from utils.logger_utils import setup_clean_logger, mute_external_loggers

logger = setup_clean_logger("tornado", logging.INFO)
mute_external_loggers(["sisepuede"])
logger.info("Environment ready.")
logger.info(f"Project dir : {cfg.PROJECT_DIR}")
logger.info(f"Run output  : {cfg.RUN_ID_OUTPUT_DIR}")

## 1 · Model initialisation

In [ ]:
from model_setup import initialize_model

_file_structure, _attr_time_period, matt, regions = initialize_model(
    y0=cfg.YEAR_START,
    y1=cfg.YEAR_END,
)
logger.info(f"Model initialised  y0={cfg.YEAR_START}  y1={cfg.YEAR_END}")

## 2 · Load run data

In [ ]:
from data_loading import load_attribute_tables, parse_strategy_metadata, load_wide_export

# Attribute tables
att_primary, att_strategy = load_attribute_tables(cfg.RUN_ID_OUTPUT_DIR)
att_strategy = parse_strategy_metadata(att_strategy)

logger.info(f"att_primary  : {att_primary.shape}")
logger.info(f"att_strategy : {att_strategy.shape}")

# Wide-format simulation output
df_export = load_wide_export(cfg.RUN_ID_OUTPUT_DIR, cfg.PRIMARY_IDS_FILTER)
logger.info(f"df_export    : {df_export.shape}")

## 3 · Post-processing — intertemporal decomposition

In [ ]:
from postprocessing import run_decomposition

df_decomposed = run_decomposition(
    df_export    = df_export,
    project_dir  = cfg.PROJECT_DIR,
    targets_path = cfg.TARGETS_PATH,
    iso_code3    = cfg.ISO_CODE3,
    year_ref     = cfg.YEAR_REF,
    region       = cfg.REGION,
    output_path  = cfg.OUTPUT_DECOMPOSED,
)
logger.info(f"df_decomposed : {df_decomposed.shape}  → {cfg.OUTPUT_DECOMPOSED}")

## 4 · Cost-benefits

In [ ]:
from cost_benefits_pipeline import run_cost_benefits

cb_data = run_cost_benefits(
    df_decomposed      = df_decomposed,
    att_primary        = att_primary,
    att_strategy       = att_strategy,
    cb_config_path     = cfg.CB_CONFIG_PATH,
    run_output_dir     = cfg.RUN_ID_OUTPUT_DIR,
    project_dir        = cfg.PROJECT_DIR,
    strategy_code_base = cfg.STRATEGY_CODE_BASE,
)
logger.info(f"cb_data : {cb_data.shape}  → {cfg.OUTPUT_CB_DATA}")

## 5 · Marginal Abatement Cost (MAC)

In [ ]:
from mac_pipeline import run_mac_analysis

mac_df = run_mac_analysis(
    df_decomposed           = df_decomposed,
    cb_data                 = cb_data,
    att_primary             = att_primary,
    att_strategy            = att_strategy,
    iso_code3               = cfg.ISO_CODE3,
    region                  = cfg.REGION,
    invent_dir              = cfg.INVENT_DIR,
    run_output_dir          = cfg.RUN_ID_OUTPUT_DIR,
    strategy_code_baseline  = cfg.STRATEGY_CODE_BASELINE,
)
logger.info(f"mac_df : {mac_df.shape}  → {cfg.OUTPUT_MAC}")

## 6 · Summary

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {"output": "decomposed SSP",    "rows": len(df_decomposed), "cols": df_decomposed.shape[1], "file": cfg.OUTPUT_DECOMPOSED.name},
    {"output": "cost-benefits data","rows": len(cb_data),       "cols": cb_data.shape[1],       "file": cfg.OUTPUT_CB_DATA.name},
    {"output": "MAC curves",        "rows": len(mac_df),        "cols": mac_df.shape[1],        "file": cfg.OUTPUT_MAC.name},
])
print(summary.to_string(index=False))